# Pandas 缺失值边界练习：金额与记录时间

## 1. 练习背景

某系统导出了一批交易记录。

本次练习只处理两个字段：

- `amount`：交易金额
- `record_time`：记录时间

数据中混入了货币符号、千位分隔符、伪缺失值、无法转换的金额文本、无效日期和无法识别的时间文本。

本练习不涉及多表关联、重复值或复杂业务规则，只训练缺失值标准化、数值转换、日期转换、问题记录保存和结果验证。

---

## 2. 字段说明

| 字段 | 含义 | 预期类型 |
|---|---|---|
| `record_id` | 原始记录编号，用于追溯 | 整数 |
| `amount` | 交易金额 | 浮点数 |
| `record_time` | 记录时间 | 日期时间 |

---

## 3. `amount` 清洗规则

`amount` 中可能存在：

- 首尾空格
- 千位分隔符 `,`
- 人民币符号 `￥`
- 空字符串
- `N/A`
- `NULL`
- `NaN`
- `-`
- Python 或 Pandas 缺失值
- 看起来像数字但实际无法转换的字符串

要求：

1. 将字段转换为 Pandas `string` 类型。
2. 删除首尾空格。
3. 删除金额中的千位分隔符。
4. 删除人民币符号。
5. 将各种伪缺失值统一转换为真正缺失值。
6. 将字段转换为数值类型。
7. 无法转换的非标准金额统一转换为缺失值。
8. 金额缺失值保留，不删除、不填充。
9. 检查转换后的数据类型、缺失数量和数值分布。
10. 检查非缺失金额中是否存在负数。

---

## 4. `record_time` 清洗规则

`record_time` 中可能存在：

- 首尾空格
- 空字符串
- `NULL`
- `not recorded`
- Python 缺失值
- 不存在的日期
- 不合法的小时

合法时间格式为：

```text
YYYY-MM-DD HH:MM
```

要求：

1. 将字段转换为 Pandas `string` 类型。
2. 删除首尾空格。
3. 将伪缺失值统一转换为真正缺失值。
4. 使用明确的时间格式转换为日期时间类型。
5. 将无法解析的时间统一转换为 `NaT`。
6. `record_time` 是关键字段。
7. 删除 `record_time` 缺失或转换失败的整行记录。
8. 删除前必须将问题记录保存为独立 DataFrame。
9. 为问题记录增加删除原因。
10. 删除后重新整理索引。

---

## 5. 缺失报告

在删除无效时间记录之后，生成字段级缺失报告：

```text
missing_report
```

缺失报告至少包含以下三列：

| 字段 | 含义 |
|---|---|
| `column` | 字段名 |
| `missing_count` | 缺失数量 |
| `missing_rate` | 缺失比例 |

要求：

- 缺失比例保留四位小数。
- 按缺失数量降序排列。
- 缺失数量相同时，按字段名升序排列。

---

## 6. 最终验证要求

使用 `assert` 验证以下条件：

1. 原始数据共有 16 行。
2. 无效时间记录共有 5 行。
3. 最终清洗数据共有 11 行。
4. `record_id` 无缺失且保持唯一。
5. `amount` 已转换为数值类型。
6. `amount` 保留 6 个缺失值。
7. 非缺失金额中不存在负数。
8. `record_time` 已转换为日期时间类型。
9. `record_time` 不再存在缺失值。
10. 保留记录数与删除记录数之和等于原始记录数。
11. 保留记录与删除记录的 `record_id` 合并后，能够完整对应原始数据。

---

## 7. 限制条件

- 不允许直接修改 `df_raw`。
- 不允许手工逐行修改数据。
- 不允许直接对整个 DataFrame 使用无条件的 `dropna()`。
- 不允许删除金额缺失的记录。
- 删除时间异常记录之前，必须先保存问题记录。
- 不允许根据清洗后的最小值和最大值自行制定金额合法范围。

In [ ]:

## 数据构造代码


import pandas as pd

data = {
    "record_id": list(range(1, 17)),

    "amount": [
        "1,299.50",
        " ￥850.00 ",
        "399.90",
        "",
        "N/A",
        None,
        "1,050.00",
        "-",
        "NaN",
        " 88.80 ",
        "2,500",
        "0",
        "320.00",
        pd.NA,
        "1,2O0.00",
        "￥600.00"
    ],

    "record_time": [
        "2026-07-03 08:00",
        " 2026-07-03 08:15 ",
        "2026-02-30 08:30",
        "2026-07-03 08:45",
        "",
        "2026-07-03 09:15",
        "not recorded",
        "2026-07-03 09:45",
        "2026-07-03 10:00",
        None,
        "2026-07-03 10:30",
        "2026-07-03 10:45",
        "2026-07-03 25:00",
        "2026-07-03 11:15",
        "2026-07-03 11:30",
        "2026-07-03 11:45"
    ]
}

df_raw = pd.DataFrame(data)

df_raw.head()

In [ ]:
# 查看原始数据规模
df_raw.shape

In [ ]:
# 查看字段类型以及非缺失值数量
df_raw.info()

In [ ]:
# 备份原始数据

df_cleaning = df_raw.copy()

### 一、清洗 `amount`

In [38]:
# ==================================================
# amount 字段清洗
# ==================================================

# 1. 备份最原始的 amount
# 用于追溯清洗前的数据，不再修改这个变量
amount_original = df_cleaning["amount"].copy()


# 2. 定义已确认的伪缺失值
# 这些内容不代表真实金额，应统一转换为 pd.NA
missing_markers = {
    "": pd.NA,
    "N/A": pd.NA,
    "n/a": pd.NA,
    "NULL": pd.NA,
    "null": pd.NA,
    "NONE": pd.NA,
    "None": pd.NA,
    "NAN": pd.NA,
    "NaN": pd.NA,
    "nan": pd.NA,
    "-": pd.NA,
    "--": pd.NA
}


# 3. 清理金额文本
amount_text = (
    df_cleaning["amount"]
    .astype("string")                         # 转为 Pandas 可空字符串类型
    .str.strip()                              # 删除首尾空格
    .str.replace(",", "", regex=False)        # 删除千位分隔符
    .str.replace("￥", "", regex=False)       # 删除人民币符号
    .replace(missing_markers)                 # 统一伪缺失值
)


# 4. 保存“数值转换前”的清理结果
# 后面需要用它判断哪些值原来有内容，但转换失败
amount_text_before_conversion = amount_text.copy()


# 5. 尝试转换为数值
# 无法转换的内容会被转换为 NaN
amount_numeric = pd.to_numeric(
    amount_text_before_conversion,
    errors="coerce"
)


# 6. 标记真正的转换失败记录
# 条件一：转换前不是缺失值
# 条件二：转换后变成了缺失值
invalid_amount_mask = (
    amount_text_before_conversion.notna()
    & amount_numeric.isna()
)


# 7. 保存转换失败的记录，便于检查和追溯
invalid_amount_rows = (
    df_cleaning.loc[
        invalid_amount_mask,
        ["record_id", "amount"]
    ]
    .copy()
)

invalid_amount_rows["invalid_reason"] = "cannot_convert_to_numeric"


# 8. 将数值转换结果写回 df_cleaning
df_cleaning["amount"] = amount_numeric


# ==================================================
# amount 字段验证
# ==================================================

# 查看转换失败的记录数量
print("转换失败数量：", invalid_amount_mask.sum())

# 查看转换失败的记录
display(invalid_amount_rows)

# 查看最终数据类型
print("amount 数据类型：", df_cleaning["amount"].dtype)

# 查看最终缺失数量
print("amount 缺失数量：", df_cleaning["amount"].isna().sum())

# 查看数值分布
display(df_cleaning["amount"].describe())


# 9. 检查非缺失金额中是否存在负数
negative_amount_mask = (
    df_cleaning["amount"].notna()
    & df_cleaning["amount"].lt(0)
)

print("负数金额数量：", negative_amount_mask.sum())

# 查看负数记录
display(
    df_cleaning.loc[
        negative_amount_mask,
        ["record_id", "amount"]
    ]
)

转换失败数量： 0


,record_id,amount,invalid_reason


amount 数据类型： Float64
amount 缺失数量： 7


count          9.0
mean         789.8
std      772.96906
min            0.0
25%          320.0
50%          600.0
75%         1050.0
max         2500.0
Name: amount, dtype: Float64

负数金额数量： 0


,record_id,amount
